## nb_03_1_player_info_silver

Cleans and validates table in bronze schema and and writes the result to sliver schema
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


StatementMeta(, 2f519e00-4a1d-4f47-80c9-ef14b0207abf, 3, Finished, Available, Finished, False)

### Load common db functions

In [2]:
%run nb_00_dbutils

StatementMeta(, 2f519e00-4a1d-4f47-80c9-ef14b0207abf, 15, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [3]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 2f519e00-4a1d-4f47-80c9-ef14b0207abf, 16, Finished, Available, Finished, False)

### Config

In [4]:
BRONZE_TABLE = "bronze.team_info"
SILVER_TABLE = "silver.team_info"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "team_id",
    "franchiseId",
    "shortName",
    "teamName",
    "abbreviation",
]

# Natural key used to de-duplicate rows
PRIMARY_KEYS: list[str] = ["team_id"]
DEDUPE_KEYS: list[str] = ["team_id"]


StatementMeta(, 2f519e00-4a1d-4f47-80c9-ef14b0207abf, 17, Finished, Available, Finished, False)

## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [5]:
if RUN_PIPELINE:
    df: DataFrame = load_table(spark, BRONZE_TABLE, columns=required_cols)

    df = df.dropDuplicates(DEDUPE_KEYS)
    df = validate_no_nulls(df, columns=required_cols)

    # Validate primary key
    df = validate_primary_keys(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 2f519e00-4a1d-4f47-80c9-ef14b0207abf, 18, Finished, Available, Finished, False)

✅ Loaded bronze.team_info: 33 rows
🏁 Silver load complete.
